In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

读取数据并整理

In [2]:
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e1'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')

TARGET = 'exam_score'
y = train[TARGET].values

X = train.drop(columns=[TARGET, 'id'])
X_test = test.drop(columns=['id'])

Train shape: (630000, 13)
Test shape: (270000, 12)


定义一个函数，根据原始特征构造出 4 个新特征，然后把这个函数同时应用到训练集 X 和测试集 X_test 上

In [3]:
def feature_engineering(df):
    df = df.copy()
    df['study_attendance'] = df['study_hours'] * df['class_attendance']
    df['study_sleep_ratio'] = df['study_hours'] / (df['sleep_hours'] + 1)
    df['attendance_study_diff'] = df['class_attendance'] - df['study_hours']
    df['sleep_hours_sq'] = df['sleep_hours'] ** 2
    return df

X = feature_engineering(X)
X_test = feature_engineering(X_test)

特征类型识别

In [4]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numerical features: {num_cols}')
print(f'Categorical features: {cat_cols}')

Numerical features: ['age', 'study_hours', 'class_attendance', 'sleep_hours', 'study_attendance', 'study_sleep_ratio', 'attendance_study_diff', 'sleep_hours_sq']
Categorical features: ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


预处理管道构建

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

交叉验证 + Ridge 基准模型 + LightGBM 残差学习 + 融合预测

In [6]:
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f'\n===== Fold {fold} =====')

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Ridge 线性模型
    X_train_prep = preprocessor.fit_transform(X_train)
    X_val_prep = preprocessor.transform(X_val)
    X_test_prep = preprocessor.transform(X_test)

    ridge = Ridge(alpha=1.0, random_state=42)
    ridge.fit(X_train_prep, y_train)

    ridge_train_pred = ridge.predict(X_train_prep)
    ridge_val_pred = ridge.predict(X_val_prep)
    ridge_test_pred = ridge.predict(X_test_prep)

    # 计算残差
    residual_train = y_train - ridge_train_pred
    residual_val = y_val - ridge_val_pred

    # LightGBM 
    X_train_lgb = X_train.copy()
    X_val_lgb = X_val.copy()
    X_test_lgb = X_test.copy()

    for col in cat_cols:
        X_train_lgb[col] = X_train_lgb[col].astype('category')
        X_val_lgb[col] = X_val_lgb[col].astype('category')
        X_test_lgb[col] = X_test_lgb[col].astype('category')

    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    lgb_model.fit(
        X_train_lgb, residual_train,
        eval_set=[(X_val_lgb, residual_val)],
        eval_metric='rmse',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )

    lgb_val_pred = lgb_model.predict(X_val_lgb)
    lgb_test_pred = lgb_model.predict(X_test_lgb)

    # 融合预测
    oof_pred[val_idx] = ridge_val_pred + lgb_val_pred
    test_pred += (ridge_test_pred + lgb_test_pred) / N_SPLITS

    # 修正后的 RMSE 计算
    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_pred[val_idx]))
    print(f'Fold {fold} RMSE: {fold_rmse:.4f}')


===== Fold 1 =====
Fold 1 RMSE: 8.7543

===== Fold 2 =====
Fold 2 RMSE: 8.7606

===== Fold 3 =====
Fold 3 RMSE: 8.7512

===== Fold 4 =====
Fold 4 RMSE: 8.7744

===== Fold 5 =====
Fold 5 RMSE: 8.7879


计算整体 OOF RMSE，打印结果

In [7]:
overall_rmse = np.sqrt(mean_squared_error(y, oof_pred))
print(f'\nOverall OOF RMSE: {overall_rmse:.4f}')


Overall OOF RMSE: 8.7657


生成提交文件

In [8]:
submission = pd.DataFrame({
    'id': test['id'],
    'exam_score': test_pred
})
submission = submission[sample_submission.columns]
submission.to_csv('submission.csv', index=False)

print('\nSubmission saved to submission.csv')
print(submission.head())


Submission saved to submission.csv
       id  exam_score
0  630000   71.814471
1  630001   70.388918
2  630002   87.745913
3  630003   55.620132
4  630004   46.780857
